# 🧬 BioRAG: Biomedical Research Assistant
This notebook implements a Retrieval-Augmented Generation (RAG) system specifically for lung cancer research papers. It includes text extraction, semantic search, and multiple LLM answering strategies.

## 1. Data Extraction
Mounts Google Drive and parses PMC XML files. Extracts title, abstract, and body text from all downloaded papers.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import xml.etree.ElementTree as ET
import re

DRIVE_BASE = '/content/drive/MyDrive/biorag'
PAPER_DIR = f'{DRIVE_BASE}/papers/lung_cancer_biomarkers'

# Extract text from all papers
def extract_text(xml_file):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        title = root.find('.//article-title')
        title_text = title.text if title is not None else ""

        abstract = root.find('.//abstract')
        abstract_text = ' '.join(abstract.itertext()) if abstract is not None else ""

        paras = []
        for p in root.findall('.//p'):
            if p.text:
                paras.append(p.text)
        body_text = ' '.join(paras)

        full_text = f"Title: {title_text}\n\nAbstract: {abstract_text}\n\n{body_text}"
        full_text = re.sub(r'\s+', ' ', full_text)

        return full_text
    except:
        return ""

# Load all papers
xml_files = list(Path(PAPER_DIR).glob('*.xml'))
papers = []
paper_names = []

for xml_file in xml_files:
    text = extract_text(xml_file)
    if len(text) > 500:
        papers.append(text)
        paper_names.append(xml_file.name)

print(f"✅ Loaded {len(papers)} papers")
print(f"📄 Papers: {', '.join(paper_names[:3])}...")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loaded 30 papers
📄 Papers: PMC13135549.xml, PMC13135499.xml, PMC13135614.xml...


## 2. LLM Initialization (Groq)
Installs the Groq client and sets up the API key securely using Colab Secrets or manual entry. Uses `llama-3.3-70b-versatile` for biomedical queries.

## 🔑 How to Set Up Your Groq API Key

### Option 1: Use Colab Secrets (Recommended)

1. Look for the **🔑 Secrets** icon in the left sidebar of Colab
2. Click **"+ Add new secret"**
3. Enter **Name:** `GROQ_API_KEY`
4. Enter **Value:** Your Groq API key (get from [console.groq.com](https://console.groq.com))
5. Toggle the notebook access button to **ON**

<img src="https://pyimagesearch.com/wp-content/uploads/2025/04/Screenshot-2025-04-02-at-4.30.54%E2%80%AFPM.png" width="500">

### Option 2: Manual Entry

If you don't want to use Secrets, you'll be prompted to enter your API key when the notebook runs.

### Get a Free Groq API Key

1. Go to [console.groq.com](https://console.groq.com)
2. Sign up for a free account
3. Navigate to **API Keys** section
4. Click **"Create API Key"**
5. Copy your key

> **Free Tier:** 5,000+ requests per month - plenty for this notebook!

In [2]:
from google.colab import userdata
from getpass import getpass

try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print("✅ Using API key from Colab Secrets")
except:
    GROQ_API_KEY = getpass("🔑 Enter your Groq API key: ")
    print("✅ API key set for this session")

✅ Using API key from Colab Secrets


In [3]:
!pip install groq -q

from groq import Groq

client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq ready")

✅ Groq ready


# 3.  LLM loading



In [4]:
# Use sentence-transformers with a small, reliable model
!pip install sentence-transformers -q

from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import drive
import xml.etree.ElementTree as ET
import re
from pathlib import Path

drive.mount('/content/drive')

# Load a lightweight model that doesn't need authentication
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')  # Small, fast, no login needed
print("✅ Model loaded! Embedding dimension:", model.get_sentence_embedding_dimension())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded! Embedding dimension: 384


/tmp/ipykernel_20365/2437767750.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("✅ Model loaded! Embedding dimension:", model.get_sentence_embedding_dimension())


## 4. Load Papers and Generate Embeddings
Loads the 30 downloaded papers, converts them into 384-dimensional vectors, and stores them for similarity search.

In [5]:
DRIVE_BASE = '/content/drive/MyDrive/biorag'
PAPER_DIR = f'{DRIVE_BASE}/papers/lung_cancer_biomarkers'

# Load papers
papers = []
paper_names = []

for xml_file in Path(PAPER_DIR).glob('*.xml'):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        title = root.find('.//article-title')
        title_text = title.text if title is not None else ""

        abstract = root.find('.//abstract')
        abstract_text = ' '.join(abstract.itertext()) if abstract is not None else ""

        # Take first 1000 chars of full text
        paras = []
        for p in root.findall('.//p'):
            if p.text:
                paras.append(p.text)
        body_text = ' '.join(paras)[:800]

        full_text = f"{title_text}\n{abstract_text}\n{body_text}"
        full_text = re.sub(r'\s+', ' ', full_text)

        if len(full_text) > 300:
            papers.append(full_text[:1500])
            paper_names.append(xml_file.name)
    except:
        pass

print(f"✅ Loaded {len(papers)} papers")

# Generate embeddings (TRUE VECTORS!)
print("Generating embeddings...")
embeddings = model.encode(papers, show_progress_bar=True)
print(f"✅ Embeddings shape: {embeddings.shape}")

✅ Loaded 30 papers
Generating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings shape: (30, 384)


## 5. Semantic Search Function
Implements cosine similarity to find papers semantically related to a user's query. Returns top-k papers with similarity scores. Loads the all-MiniLM-L6-v2 sentence transformer model to convert papers into dense vector embeddings for semantic search.Implements cosine similarity to find papers semantically related to a user's query. Returns top-k papers with similarity scores.

In [6]:
def semantic_search(query, embeddings, papers, paper_names, top_k=3):
    """True semantic search using cosine similarity"""

    # Embed the query
    query_embedding = model.encode([query])[0]

    # Calculate semantic similarities
    similarities = cosine_similarity([query_embedding], embeddings)[0]

    # Get top matches
    top_indices = similarities.argsort()[-top_k:][::-1]

    results = []
    for idx in top_indices:
        results.append({
            'paper': paper_names[idx],
            'text': papers[idx],
            'similarity': float(similarities[idx])
        })

    return results

# Test semantic search
test_query = "genetic mutations and biomarkers for lung cancer diagnosis"
results = semantic_search(test_query, embeddings, papers, paper_names)

print(f"🔍 Semantic Search for: '{test_query}'\n")
for r in results:
    print(f"📄 {r['paper']} (Similarity: {r['similarity']:.3f})")
    print(f"   Excerpt: {r['text'][:150]}...\n")

🔍 Semantic Search for: 'genetic mutations and biomarkers for lung cancer diagnosis'

📄 PMC13135135.xml (Similarity: 0.575)
   Excerpt: Age‐Associated Targetable Genomic Alterations and ABSTRACT Aim To investigate the landscape of targetable genomic alterations and programmed cell deat...

📄 PMC13135767.xml (Similarity: 0.351)
   Excerpt: Multi-Omics Characterization of ABHD12 Across Pan-Cancer and Validation of Its Role in Promoting Proliferation and Metastasis in Breast Cancer Purpose...

📄 PMC13134949.xml (Similarity: 0.344)
   Excerpt: Identification of Critical ABSTRACT Background Oral squamous cell carcinoma (OSCC) is marked by frequent recurrence rates and an unclear etiology, und...



## 6. Semantic RAG (Retrieval-Augmented Generation)
Complete RAG pipeline: retrieves semantically relevant papers, builds context, and generates answers using Llama 3.3 model.This is a proper RAG pipeline. It uses vector similarity to find the most relevant chunks of text first, then feeds only those specific sources to the LLM for a high-accuracy answer

In [7]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

def true_rag(question):
    """Complete True RAG pipeline"""

    # Step 1: Semantic retrieval (this is the R in RAG)
    retrieved = semantic_search(question, embeddings, papers, paper_names, top_k=3)

    # Step 2: Build context from semantically similar papers
    context = ""
    for r in retrieved:
        context += f"\n[SOURCE: {r['paper']}] [RELEVANCE: {r['similarity']:.2f}]\n{r['text']}\n"

    # Step 3: Generate answer (this is the G in RAG)
    prompt = f"""You are a biomedical expert. Answer using ONLY the semantically retrieved papers below.

RETRIEVED PAPERS (by vector similarity):
{context}

QUESTION: {question}

INSTRUCTIONS:
- Base answer ONLY on the retrieved papers
- Cite which paper provided each fact
- Include specific numbers, mutations, treatments if mentioned

ANSWER:"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=500
    )

    return response.choices[0].message.content, retrieved

# Test TRUE RAG
question = "What are the common genetic mutations in lung cancer?"
answer, sources = true_rag(question)

print("="*60)
print("🔬 TRUE RAG - Semantic Vector Search + LLM")
print("="*60)
print(f"\n📝 QUESTION: {question}\n")
print(f"💡 ANSWER:\n{answer}\n")
print("📚 SEMANTICALLY RETRIEVED SOURCES:")
for s in sources:
    print(f"   • {s['paper']} (vector similarity: {s['similarity']:.3f})")

🔬 TRUE RAG - Semantic Vector Search + LLM

📝 QUESTION: What are the common genetic mutations in lung cancer?

💡 ANSWER:
According to the retrieved papers, the common genetic mutations in lung cancer, specifically in pulmonary ground-glass opacities (GGOs), include:

* EGFR mutations, which have a relatively high mutation rate of 61.5% [PMC13135135.xml]
* ERBB2 mutations, which have a relatively high mutation rate of 12.0% [PMC13135135.xml]
* KRAS mutations, which have a relatively low mutation rate of 8.2% [PMC13135135.xml]
* ALK rearrangements, which have a relatively low mutation rate of 2.3% [PMC13135135.xml]
* MET mutations, which increase significantly with age [PMC13135135.xml]
* RET rearrangements, which decrease significantly with age [PMC13135135.xml]

It's worth noting that the papers do not provide a comprehensive list of all common genetic mutations in lung cancer, but rather focus on specific mutations and their association with age and other factors. Additionally, the pap

In [10]:
# ============================================
# BRAND NEW CLIENT WITH FRESH AUTH
# ============================================

import groq

# Get key fresh
from google.colab import userdata
FRESH_API_KEY = userdata.get('GROQ_API_KEY')

# Create brand new client
fresh_client = groq.Groq(api_key=FRESH_API_KEY)

def true_rag_v2(question):
    """Complete True RAG pipeline with fresh client"""

    retrieved = semantic_search(question, embeddings, papers, paper_names, top_k=3)

    context = ""
    for r in retrieved:
        context += f"\n[SOURCE: {r['paper']}] [RELEVANCE: {r['similarity']:.2f}]\n{r['text']}\n"

    prompt = f"""You are a biomedical expert. Answer using ONLY the semantically retrieved papers below.

RETRIEVED PAPERS (by vector similarity):
{context}

QUESTION: {question}

ANSWER:"""

    response = fresh_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=500
    )

    return response.choices[0].message.content, retrieved

# Test
question = "What are the common genetic mutations in lung cancer?"
answer, sources = true_rag_v2(question)

print("="*60)
print("🔬 TRUE RAG - Semantic Vector Search + LLM")
print("="*60)
print(f"\n📝 QUESTION: {question}\n")
print(f"💡 ANSWER:\n{answer}\n")
print("📚 SEMANTICALLY RETRIEVED SOURCES:")
for s in sources:
    print(f"   • {s['paper']} (vector similarity: {s['similarity']:.3f})")

APIConnectionError: Connection error.

## 7. 3-Way Comparison RAG System
Compares three different answering approaches side-by-side:

- **Raw Retrieval** - Shows search results with similarity scores (no LLM)
- **Standard LLM** - Direct AI answer from retrieved papers
- **Few-shot LLM** - Structured answer guided by examples

In [8]:
# ============================================
# 3-WAY COMPARISON RAG SYSTEM
# ============================================

from google.colab import drive
drive.mount('/content/drive')

!pip install sentence-transformers groq -q

from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import xml.etree.ElementTree as ET
import re
from pathlib import Path
from groq import Groq
import pandas as pd

# ============================================
# LOAD EVERYTHING
# ============================================
DRIVE_BASE = '/content/drive/MyDrive/biorag'
PAPER_DIR = f'{DRIVE_BASE}/papers/lung_cancer_biomarkers'

print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Loading papers...")
papers = []
paper_names = []

for xml_file in Path(PAPER_DIR).glob('*.xml'):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        title = root.find('.//article-title')
        title_text = title.text if title is not None else ""
        abstract = root.find('.//abstract')
        abstract_text = ' '.join(abstract.itertext()) if abstract is not None else ""

        paras = [p.text for p in root.findall('.//p') if p.text]
        body_text = ' '.join(paras)[:800]

        full_text = f"{title_text}\n{abstract_text}\n{body_text}"
        full_text = re.sub(r'\s+', ' ', full_text)

        if len(full_text) > 300:
            papers.append(full_text[:1500])
            paper_names.append(xml_file.name)
    except:
        pass

print(f"✅ Loaded {len(papers)} papers")

# Generate embeddings
embeddings = model.encode(papers, show_progress_bar=True)

# Initialize Groq
client = Groq(api_key=GROQ_API_KEY)

# ============================================
# SEMANTIC SEARCH FUNCTION
# ============================================
def semantic_search(query, top_k=3):
    """Retrieve top-k semantically similar papers"""
    query_embedding = model.encode([query])[0]
    similarities = cosine_similarity([query_embedding], embeddings)[0]
    top_indices = similarities.argsort()[-top_k:][::-1]

    results = []
    for i in top_indices:
        results.append({
            'paper': paper_names[i],
            'text': papers[i],
            'similarity': float(similarities[i])
        })
    return results

# ============================================
# THREE DIFFERENT APPROACHES
# ============================================

def approach_1_raw_retrieval(question, top_k=3):
    """Column 1: Just show retrieved sentences, no LLM"""
    retrieved = semantic_search(question, top_k)

    output = f"\n{'='*60}\n"
    output += f"📥 RAW RETRIEVAL (No LLM - Just Search Results)\n"
    output += f"Question: {question}\n"
    output += f"{'='*60}\n\n"

    for i, r in enumerate(retrieved, 1):
        output += f"[Result {i}] Paper: {r['paper']} (Similarity: {r['similarity']:.3f})\n"
        output += f"Excerpt: {r['text'][:400]}...\n\n"
        output += f"{'-'*40}\n\n"

    return output, retrieved

def approach_2_standard_llm(question, retrieved):
    """Column 2: Standard LLM answer without examples"""
    context = "\n".join([f"[{r['paper']}]\n{r['text']}" for r in retrieved])

    prompt = f"""Answer the question based ONLY on the retrieved papers below.

RETRIEVED PAPERS:
{context}

QUESTION: {question}

ANSWER (be direct, concise, and cite sources):"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=500
    )

    output = f"\n{'='*60}\n"
    output += f"🤖 STANDARD LLM (Direct Answer)\n"
    output += f"Question: {question}\n"
    output += f"{'='*60}\n\n"
    output += response.choices[0].message.content
    output += f"\n\n{'='*60}\n"

    return output

def approach_3_few_shot_llm(question, retrieved):
    """Column 3: Few-shot learning with examples"""
    context = "\n".join([f"[{r['paper']}]\n{r['text']}" for r in retrieved])

    # Few-shot examples to guide the model
    examples = """
EXAMPLE 1:
Question: "What is the mutation rate of EGFR in lung cancer?"
Answer: "Based on PMC13135135.xml, EGFR mutation rate is 61.5% in pulmonary ground-glass opacities."

EXAMPLE 2:
Question: "Which mutations have the highest frequency?"
Answer: "According to PMC13135135.xml: EGFR (61.5%) has the highest frequency, followed by ERBB2 (12.0%), then KRAS (8.2%)."

EXAMPLE 3:
Question: "What treatments are mentioned?"
Answer: "The papers mention targeted therapies for EGFR-mutant patients, but specific drug names are not provided in the retrieved excerpts."
"""

    prompt = f"""You are a biomedical expert. Answer the question based ONLY on the retrieved papers below. Follow the example format.

EXAMPLE FORMATS (use these as guides):
{examples}

RETRIEVED PAPERS:
{context}

QUESTION: {question}

ANSWER (follow the example style - be specific, cite sources, include numbers where available):"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=500
    )

    output = f"\n{'='*60}\n"
    output += f"🎯 FEW-SHOT LLM (With Examples)\n"
    output += f"Question: {question}\n"
    output += f"{'='*60}\n\n"
    output += response.choices[0].message.content
    output += f"\n\n{'='*60}\n"

    return output

# ============================================
# COMPLETE 3-WAY COMPARISON
# ============================================
def compare_approaches(question):
    """Run all three approaches and display comparison"""

    print("\n" + "🔍" * 30)
    print(f"COMPARING 3 APPROACHES FOR: '{question}'")
    print("🔍" * 30)

    # Approach 1: Raw Retrieval
    raw_output, retrieved = approach_1_raw_retrieval(question)
    print(raw_output)

    # Approach 2: Standard LLM
    standard_output = approach_2_standard_llm(question, retrieved)
    print(standard_output)

    # Approach 3: Few-shot LLM
    fewshot_output = approach_3_few_shot_llm(question, retrieved)
    print(fewshot_output)

    # Summary Table
    print("\n" + "📊" * 30)
    print("SUMMARY COMPARISON")
    print("📊" * 30)
    print(f"""
    | Approach | What It Does | Best For |
    |----------|--------------|----------|
    | Raw Retrieval | Shows search results only | Transparency, verifying sources |
    | Standard LLM | Direct answer from context | Quick, straightforward answers |
    | Few-shot LLM | Formatted answer with examples | Structured, consistent outputs |
    """)

    return retrieved

# ============================================
# TEST THE 3-WAY COMPARISON
# ============================================
question = "What are the common genetic mutations in lung cancer?"
retrieved = compare_approaches(question)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading papers...
✅ Loaded 30 papers


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍
COMPARING 3 APPROACHES FOR: 'What are the common genetic mutations in lung cancer?'
🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍

📥 RAW RETRIEVAL (No LLM - Just Search Results)
Question: What are the common genetic mutations in lung cancer?

[Result 1] Paper: PMC13135135.xml (Similarity: 0.534)
Excerpt: Age‐Associated Targetable Genomic Alterations and ABSTRACT Aim To investigate the landscape of targetable genomic alterations and programmed cell death ligand 1 (PD‐L1) expression in pulmonary ground‐glass opacities (GGOs) and their association with age. Methods A total of 2509 patients with GGOs were retrospectively analyzed. Tumor characteristics, PD‐L1 expression, and prevalence of targetable a...

----------------------------------------

[Result 2] Paper: PMC13135767.xml (Similarity: 0.291)
Excerpt: Multi-Omics Characterization of ABHD12 Across Pan-Cancer and Validation of Its Role in Promoting Proliferation and Metastasis in Breast Cancer Purpose ABHD12 is linked

## 8. Interactive 3-Way Comparison
Launches an interactive Q&A loop. Enter any biomedical question to see all three approaches compared in real-time.

In [9]:
# ============================================
# INTERACTIVE 3-WAY COMPARISON
# ============================================

print("\n" + "="*70)
print("🔬 3-WAY RAG COMPARISON SYSTEM")
print("="*70)
print("""
This system shows you THREE different outputs for each question:

1️⃣ RAW RETRIEVAL - Shows what the search found (no AI)
2️⃣ STANDARD LLM - Direct AI answer from papers
3️⃣ FEW-SHOT LLM - AI answer guided by examples

Compare them to see which works best for your needs!
""")

while True:
    question = input("\n❓ Ask a biomedical question (or 'quit'): ").strip()

    if question.lower() == 'quit':
        print("Goodbye!")
        break

    if not question:
        continue

    compare_approaches(question)

    print("\n" + "💡" * 30)
    print("Which approach was most useful?")
    print("  • Raw Retrieval = Most transparent")
    print("  • Standard LLM = Most direct")
    print("  • Few-shot LLM = Most structured")
    print("💡" * 30)


🔬 3-WAY RAG COMPARISON SYSTEM

This system shows you THREE different outputs for each question:

1️⃣ RAW RETRIEVAL - Shows what the search found (no AI)
2️⃣ STANDARD LLM - Direct AI answer from papers
3️⃣ FEW-SHOT LLM - AI answer guided by examples

Compare them to see which works best for your needs!


❓ Ask a biomedical question (or 'quit'): what is responsible for lung cancer

🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍
COMPARING 3 APPROACHES FOR: 'what is responsible for lung cancer'
🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍

📥 RAW RETRIEVAL (No LLM - Just Search Results)
Question: what is responsible for lung cancer

[Result 1] Paper: PMC13135135.xml (Similarity: 0.393)
Excerpt: Age‐Associated Targetable Genomic Alterations and ABSTRACT Aim To investigate the landscape of targetable genomic alterations and programmed cell death ligand 1 (PD‐L1) expression in pulmonary ground‐glass opacities (GGOs) and their association with age. Methods A total of 2509 patients with GGOs were retrospectively analyzed. T